# 06 - Auditability: Calibration and Discriminative Power (Simulation)

This notebook implements the audit/validation framework for Chapter 5 on the
**simulated** panel, following the CEBS/EBA-adapted criteria: calibration,
discriminative power, and stability (Section: Evaluation).

Scope of this first pass, per discussion with Ce:

- **Calibration** - (1) mean-bias test with HAC/Newey-West standard errors,
  against both the true quarterly PD and the noisy realised loss proxy;
  (2) PIT/KS test using the full Vasicek loss distribution.
- **Discriminative power** - Kendall's tau between predicted and true quarterly
  PD, with block-bootstrap significance (simulation setting only; the empirical
  notebook will instead use NBER-based AUROC).
- **Stability** is *not* implemented yet. It needs rolling-window coefficient
  tracking for the linear models and fixed-input drift tracking for the MLPs,
  which means changes to the retraining loops in `02`/`04` to persist
  parameters/predictions at each refit - left for a follow-up notebook.

**Coverage caveat carried over from the write-up:** the black-box benchmark
never produces a PD estimate, only an allocation `alpha_hat`. It therefore
cannot be submitted to either calibration test or the discriminative-power
test here - only a stability-of-output-mapping analogue would apply to it
later. This notebook only audits models that emit `pd_q_hat`: PTO's
`linear`/`mlp`/`persistence`/`persistence_ma4` and DFL's `dfl_linear`/`dfl_mlp`.


## Part 1 - Imports, paths, and inputs

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.stats import kendalltau, norm

pd.set_option("display.float_format", "{:.6f}".format)

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "simulation" / "data" / "simulated_pd_panel.csv").exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT))
from minou_colors import MINOU_COLORS

METADATA_PATH = ROOT / "simulation" / "data" / "simulated_pd_panel_metadata.json"

PTO_RESULTS_DIR = ROOT / "simulation" / "results" / "02_pto"
BB_RESULTS_DIR = ROOT / "simulation" / "results" / "03_blackbox"
DFL_RESULTS_DIR = ROOT / "simulation" / "results" / "04_dfl"

FIGURES_DIR = ROOT / "simulation" / "figures" / "06_auditability_sim"
RESULTS_DIR = ROOT / "simulation" / "results" / "06_auditability"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Which split to audit. All three upstream notebooks currently only produce "main".
SPLIT_NAME = "main"

print(f"Project root: {ROOT}")
print(f"PTO results: {PTO_RESULTS_DIR.relative_to(ROOT)}")
print(f"Black-box results: {BB_RESULTS_DIR.relative_to(ROOT)} (no PD estimate - not audited here)")
print(f"DFL results: {DFL_RESULTS_DIR.relative_to(ROOT)}")
print(f"Figures: {FIGURES_DIR.relative_to(ROOT)}")
print(f"Results: {RESULTS_DIR.relative_to(ROOT)}")


: 

## Part 2 - Configuration

Constants and Basel/Vasicek helper functions are duplicated here (rather than
imported) so this notebook can audit any model's saved predictions
standalone, without depending on which upstream notebook was last run. They
must stay numerically identical to `01`/`02`/`03`/`04` - the cross-check in
Part 3 verifies this against the panel's own oracle columns.

In [ ]:
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    META = json.load(f)
CONST = META["constants"]

LGD = float(CONST["LGD"])
RHO_SCALE = float(CONST.get("RHO_SCALE", 1.0))
assert LGD == 0.45, "Panel constants differ from expected calibration."

PD_Q_CLIP_LOW = 1e-6
PD_Q_CLIP_HIGH = 0.99
PD_A_CLIP_LOW = 1e-6
PD_A_CLIP_HIGH = 0.99

DATE_COL = "date"

# Model families and their per-model column stems, matching the wide CSVs
# saved by 02_pto_simulated_panel and 04_dfl_simulated_panel. Black-box is
# deliberately excluded - see the scope note in Part 1.
PTO_MODELS = ["linear", "mlp", "persistence", "persistence_ma4"]
DFL_MODELS = ["dfl_linear", "dfl_mlp"]

# HAC lag and block-bootstrap defaults; kept as named constants so they are
# easy to sensitivity-check from one place.
NW_LAGS = 4
BLOCK_LENGTH = 8
N_BOOT = 2000
BOOT_SEED = 42


def clip_pd_q(pd_q):
    return np.clip(np.asarray(pd_q, dtype=float), PD_Q_CLIP_LOW, PD_Q_CLIP_HIGH)


def clip_pd_a(pd_a):
    return np.clip(np.asarray(pd_a, dtype=float), PD_A_CLIP_LOW, PD_A_CLIP_HIGH)


def quarterly_pd_to_annual(pd_q):
    pd_q = clip_pd_q(pd_q)
    return clip_pd_a(1.0 - np.power(1.0 - pd_q, 4.0))


def basel_r_from_annual_pd(pd_a):
    pd_a = clip_pd_a(pd_a)
    exp_term = (1.0 - np.exp(-50.0 * pd_a)) / (1.0 - np.exp(-50.0))
    return 0.12 * exp_term + 0.24 * (1.0 - exp_term)


def vasicek_rho(pd_a):
    # Same correlation the simulator uses for the loss draw (Basel R times RHO_SCALE),
    # matching optimise_alpha_expected in 02_pto_simulated_panel.
    return np.clip(RHO_SCALE * basel_r_from_annual_pd(pd_a), 1e-4, 0.999)


def vasicek_loss_cdf(x, pd_q, rho):
    '''F_L(x; PD, rho) for the Vasicek portfolio loss rate L ~ Vasicek(PD, rho):

        F_L(x; PD, rho) = Phi( [sqrt(1 - rho) * Phi^-1(x) - Phi^-1(PD)] / sqrt(rho) )

    Used for the PIT calibration test (Eq. u_t = F_L(L_t; PD_hat_t, rho)).'''
    x = clip_pd_q(x)
    pd_q = clip_pd_q(pd_q)
    rho = np.clip(np.asarray(rho, dtype=float), 1e-4, 0.999)
    z = (np.sqrt(1.0 - rho) * norm.ppf(x) - norm.ppf(pd_q)) / np.sqrt(rho)
    return norm.cdf(z)


def newey_west_lrv(d, lags):
    # Identical to the helper in 02_pto_simulated_panel: returns the HAC estimate
    # of Var(mean(d)), i.e. the long-run variance already scaled by 1/n.
    d = np.asarray(d, dtype=float)
    d = d - d.mean()
    n = len(d)
    v = float(np.mean(d * d))
    for k in range(1, min(lags, n - 1) + 1):
        w = 1.0 - k / (lags + 1.0)
        v += 2.0 * w * float(np.mean(d[k:] * d[:-k]))
    return v / n


print(f"Constants: LGD={LGD}, RHO_SCALE={RHO_SCALE}")
print(f"Audited PD models -> PTO: {PTO_MODELS}; DFL: {DFL_MODELS}")
print(f"HAC lags={NW_LAGS}, block length={BLOCK_LENGTH}, bootstrap reps={N_BOOT}")


: 

## Part 3 - Load saved predictions and build a long panel of PD forecasts

Reads the wide prediction CSVs saved by `02_pto_simulated_panel` and
`04_dfl_simulated_panel`, and reshapes each model's `pd_q_hat_<model>` column
into a common long frame with the shared targets (`pd_q_true`,
`pd_q_realised` = the realised loss proxy `L_t = c_t^realised / LGD`).

In [ ]:
def load_wide_predictions(results_dir, split_name, filename_stem):
    path = results_dir / f"{filename_stem}_{split_name}.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"Missing predictions file: {path}. Run the corresponding notebook first."
        )
    df = pd.read_csv(path)
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    return df.sort_values(DATE_COL).reset_index(drop=True)


def stack_pd_models(wide_df, model_names, source_label):
    records = []
    for m in model_names:
        col = f"pd_q_hat_{m}"
        if col not in wide_df.columns:
            print(f"  [skip] {source_label}:{m} - no '{col}' column in the saved predictions.")
            continue
        records.append(pd.DataFrame({
            "date": wide_df[DATE_COL],
            "model": m,
            "source": source_label,
            "pd_q_hat": wide_df[col],
            "pd_q_true": wide_df["pd_q_true"],
            "pd_q_realised": wide_df["pd_q_realised"],
            "c_q_realised": wide_df["c_q_realised"],
        }))
    return pd.concat(records, ignore_index=True) if records else pd.DataFrame()


pto_wide = load_wide_predictions(PTO_RESULTS_DIR, SPLIT_NAME, "pto_sim_predictions")
dfl_wide = load_wide_predictions(DFL_RESULTS_DIR, SPLIT_NAME, "dfl_sim_predictions")

print("Stacking PD-emitting models into a long audit panel...")
long_pto = stack_pd_models(pto_wide, PTO_MODELS, "pto")
long_dfl = stack_pd_models(dfl_wide, DFL_MODELS, "dfl")
audit_long = pd.concat([long_pto, long_dfl], ignore_index=True)

# Derived quantities needed by the calibration tests below.
audit_long["pd_a_hat"] = quarterly_pd_to_annual(audit_long["pd_q_hat"])
audit_long["rho_hat"] = vasicek_rho(audit_long["pd_a_hat"])

MODEL_ORDER = PTO_MODELS + DFL_MODELS
assert set(audit_long["model"].unique()) == set(MODEL_ORDER), "Unexpected model set in the audit panel."

print(f"Audit panel rows: {len(audit_long)} across models {MODEL_ORDER}")
display(audit_long.groupby("model").size().rename("n_obs").to_frame())
display(audit_long.head())


## Part 4 - Calibration: mean-bias test with HAC standard errors

Tests $H_0: \mathbb{E}[\widehat{PD}_t - L_t] = 0$ for each model, against two
calibration targets: the true quarterly PD (isolates pure estimation error,
simulation-only) and the noisy realised loss proxy `pd_q_realised`
(available in both simulation and empirically). HAC/Newey-West standard
errors account for serial correlation in the forecast-error series.

In [ ]:
def mean_bias_test(pd_hat, target, nw_lags=NW_LAGS):
    d = np.asarray(pd_hat, dtype=float) - np.asarray(target, dtype=float)
    n = len(d)
    bias = float(np.mean(d))
    lrv = newey_west_lrv(d, nw_lags)  # HAC estimate of Var(mean(d))
    if not np.isfinite(lrv) or lrv <= 0:
        return dict(bias=bias, hac_se=float("nan"), t_stat=float("nan"),
                    p_value=float("nan"), n=n)
    hac_se = float(np.sqrt(lrv))
    t_stat = bias / hac_se
    p_value = 2.0 * stats.t.sf(abs(t_stat), df=n - 1)
    return dict(bias=bias, hac_se=hac_se, t_stat=float(t_stat), p_value=float(p_value), n=n)


bias_rows = []
for model_name, g in audit_long.groupby("model"):
    g = g.sort_values("date")
    vs_true = mean_bias_test(g["pd_q_hat"], g["pd_q_true"])
    vs_realised = mean_bias_test(g["pd_q_hat"], g["pd_q_realised"])
    bias_rows.append({
        "model": model_name,
        "n": vs_true["n"],
        "bias_vs_true": vs_true["bias"],
        "hac_se_vs_true": vs_true["hac_se"],
        "t_stat_vs_true": vs_true["t_stat"],
        "p_value_vs_true": vs_true["p_value"],
        "bias_vs_realised": vs_realised["bias"],
        "hac_se_vs_realised": vs_realised["hac_se"],
        "t_stat_vs_realised": vs_realised["t_stat"],
        "p_value_vs_realised": vs_realised["p_value"],
    })

bias_table = pd.DataFrame(bias_rows).set_index("model").loc[MODEL_ORDER]
display(bias_table)
bias_table.to_csv(RESULTS_DIR / f"calibration_bias_test_{SPLIT_NAME}.csv")
print(f"Saved: {(RESULTS_DIR / f'calibration_bias_test_{SPLIT_NAME}.csv').relative_to(ROOT)}")


## Part 5 - Calibration: PIT / Kolmogorov-Smirnov test

For each quarter, the probability integral transform
$u_t = F_L(L_t; \widehat{PD}_t, \rho_t)$ is computed using the full Vasicek
loss CDF, with $\rho_t$ the model's own implied correlation
(`vasicek_rho(pd_a_hat_t)`, i.e. the same rho the PTO/DFL decision layers use
- not the DGP's true rho). If the model is correctly specified, the $u_t$
series should be uniform on $[0,1]$; a KS test checks this directly and is
more sensitive to dispersion/tail misspecification than the mean-bias test
alone.

In [ ]:
def pit_ks_test(pd_hat, rho_hat, realised_loss):
    u = vasicek_loss_cdf(realised_loss, pd_hat, rho_hat)
    ks_stat, ks_p = stats.kstest(u, "uniform")
    return u, ks_stat, ks_p


pit_rows = []
pit_by_model = {}
for model_name, g in audit_long.groupby("model"):
    g = g.sort_values("date")
    u, ks_stat, ks_p = pit_ks_test(g["pd_q_hat"], g["rho_hat"], g["pd_q_realised"])
    pit_by_model[model_name] = pd.DataFrame({"date": g["date"].to_numpy(), "pit": u})
    pit_rows.append({
        "model": model_name,
        "n": len(u),
        "pit_mean": float(np.mean(u)),
        "pit_std": float(np.std(u)),
        "ks_stat": float(ks_stat),
        "ks_p_value": float(ks_p),
    })

pit_table = pd.DataFrame(pit_rows).set_index("model").loc[MODEL_ORDER]
display(pit_table)
pit_table.to_csv(RESULTS_DIR / f"calibration_pit_ks_test_{SPLIT_NAME}.csv")

pit_long = pd.concat(
    [df.assign(model=m) for m, df in pit_by_model.items()], ignore_index=True
)[["date", "model", "pit"]]
pit_long.to_csv(RESULTS_DIR / f"calibration_pit_values_{SPLIT_NAME}.csv", index=False)
print(f"Saved: {(RESULTS_DIR / f'calibration_pit_ks_test_{SPLIT_NAME}.csv').relative_to(ROOT)}")
print(f"Saved: {(RESULTS_DIR / f'calibration_pit_values_{SPLIT_NAME}.csv').relative_to(ROOT)}")


## Part 6 - Discriminative power: Kendall's tau (simulation)

The true conditional PD is known in simulation, so discriminative power is
assessed directly as the rank concordance between $\widehat{PD}_t$ and
$PD_t^{\mathrm{true}}$. Significance uses a moving-block bootstrap (block
length `BLOCK_LENGTH`) rather than the asymptotic Kendall's-tau p-value,
since the underlying series is serially correlated; the analytical p-value
is reported alongside as a cross-check, not as the primary inference.

In [ ]:
def moving_block_bootstrap_tau(x, y, block_length=BLOCK_LENGTH, n_boot=N_BOOT, seed=BOOT_SEED):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    n_blocks = int(np.ceil(n / block_length))
    rng = np.random.default_rng(seed)
    starts = np.arange(n - block_length + 1)
    taus = np.empty(n_boot)
    for b in range(n_boot):
        chosen = rng.choice(starts, size=n_blocks, replace=True)
        idx = np.concatenate([np.arange(s, s + block_length) for s in chosen])[:n]
        taus[b] = kendalltau(x[idx], y[idx]).statistic
    return taus


tau_rows = []
tau_boot_by_model = {}
for model_name, g in audit_long.groupby("model"):
    g = g.sort_values("date")
    tau_res = kendalltau(g["pd_q_hat"], g["pd_q_true"])
    boot_taus = moving_block_bootstrap_tau(g["pd_q_hat"].to_numpy(), g["pd_q_true"].to_numpy())
    tau_boot_by_model[model_name] = boot_taus
    ci_low, ci_high = np.percentile(boot_taus, [2.5, 97.5])
    # Two-sided bootstrap p-value: proportion of the bootstrap null-centred
    # distribution at least as extreme as the observed statistic in magnitude.
    centred = boot_taus - np.mean(boot_taus) + 0.0
    boot_p = float(np.mean(np.abs(centred) >= abs(tau_res.statistic)))
    tau_rows.append({
        "model": model_name,
        "n": len(g),
        "tau": float(tau_res.statistic),
        "asymptotic_p_value": float(tau_res.pvalue),
        "boot_ci_low": float(ci_low),
        "boot_ci_high": float(ci_high),
        "boot_p_value": boot_p,
        "excludes_zero": bool(ci_low > 0 or ci_high < 0),
    })

tau_table = pd.DataFrame(tau_rows).set_index("model").loc[MODEL_ORDER]
display(tau_table)
tau_table.to_csv(RESULTS_DIR / f"discriminative_power_kendall_tau_{SPLIT_NAME}.csv")
print(f"Saved: {(RESULTS_DIR / f'discriminative_power_kendall_tau_{SPLIT_NAME}.csv').relative_to(ROOT)}")


## Part 7 - Combined audit summary

One row per model with the headline statistic from each test implemented so
far. Stability columns are left as placeholders (`NaN`) until that test is
built.

In [ ]:
summary = (
    bias_table[["n", "bias_vs_true", "p_value_vs_true", "bias_vs_realised", "p_value_vs_realised"]]
    .join(pit_table[["ks_stat", "ks_p_value"]])
    .join(tau_table[["tau", "boot_p_value"]])
)
summary["stability_test"] = np.nan  # placeholder - not yet implemented
display(summary)
summary.to_csv(RESULTS_DIR / f"audit_summary_{SPLIT_NAME}.csv")
print(f"Saved: {(RESULTS_DIR / f'audit_summary_{SPLIT_NAME}.csv').relative_to(ROOT)}")


## Part 8 - Diagnostic plots

In [ ]:
plt.rcdefaults()
plt.rcParams.update({"font.size": 10, "axes.linewidth": 0.8, "xtick.direction": "out",
                     "ytick.direction": "out", "xtick.major.size": 3, "ytick.major.size": 3,
                     "font.sans-serif": ["DejaVu Sans"]})

MODEL_COLORS = {
    "linear": MINOU_COLORS["teal"],
    "mlp": MINOU_COLORS["coral"],
    "persistence": MINOU_COLORS["grey"],
    "persistence_ma4": MINOU_COLORS["gold"],
    "dfl_linear": MINOU_COLORS["navy"],
    "dfl_mlp": MINOU_COLORS["sage"],
}
MODEL_LABELS = {
    "linear": "PTO Linear", "mlp": "PTO MLP",
    "persistence": "Persistence (raw draw)", "persistence_ma4": "Persistence (4Q mean)",
    "dfl_linear": "DFL Linear", "dfl_mlp": "DFL MLP",
}


def format_date_axis(ax, dates):
    years = pd.to_datetime(dates).dt.year
    step = max(2, int(np.ceil((years.max() - years.min()) / 10)))
    ax.xaxis.set_major_locator(mdates.YearLocator(step))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(True, alpha=0.28)


# 1. PIT histograms per model, faceted.
n_models = len(MODEL_ORDER)
fig, axes = plt.subplots(1, n_models, figsize=(2.6 * n_models, 3), sharey=True)
for ax, m in zip(np.atleast_1d(axes), MODEL_ORDER):
    u = pit_by_model[m]["pit"].to_numpy()
    ax.hist(u, bins=12, range=(0, 1), color=MODEL_COLORS[m], alpha=0.85, density=True)
    ax.axhline(1.0, color="black", lw=0.8, ls="--")
    ax.set_title(MODEL_LABELS[m], fontsize=9)
    ax.set_xlim(0, 1)
fig.suptitle(f"PIT histograms vs. Uniform(0,1) ({SPLIT_NAME} split)", y=1.03)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pit_histograms.png", dpi=200, bbox_inches="tight")
plt.show()

# 2. Bias time series: pd_q_hat - pd_q_true per model.
fig, ax = plt.subplots(figsize=(10, 4))
for m in MODEL_ORDER:
    g = audit_long[audit_long["model"] == m].sort_values("date")
    ax.plot(g["date"], g["pd_q_hat"] - g["pd_q_true"], color=MODEL_COLORS[m], lw=1.1, label=MODEL_LABELS[m])
ax.axhline(0.0, color="black", lw=0.8)
ax.set_ylabel("PD forecast error (hat - true)")
ax.set_title(f"Forecast error vs. true quarterly PD ({SPLIT_NAME} split)")
format_date_axis(ax, audit_long["date"])
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "bias_time_series.png", dpi=200, bbox_inches="tight")
plt.show()

# 3. Predicted vs. true PD scatter, one panel per model, for the Kendall's tau result.
fig, axes = plt.subplots(1, n_models, figsize=(2.6 * n_models, 2.8), sharex=True, sharey=True)
for ax, m in zip(np.atleast_1d(axes), MODEL_ORDER):
    g = audit_long[audit_long["model"] == m]
    ax.scatter(g["pd_q_true"], g["pd_q_hat"], s=6, alpha=0.5, color=MODEL_COLORS[m])
    lims = [min(g["pd_q_true"].min(), g["pd_q_hat"].min()), max(g["pd_q_true"].max(), g["pd_q_hat"].max())]
    ax.plot(lims, lims, color="black", lw=0.8, ls="--")
    tau_val = tau_table.loc[m, "tau"]
    ax.set_title(f"{MODEL_LABELS[m]}\n(tau={tau_val:.3f})", fontsize=9)
    ax.set_xlabel("True PD")
axes_flat = np.atleast_1d(axes)
axes_flat[0].set_ylabel("Predicted PD")
fig.suptitle(f"Predicted vs. true quarterly PD ({SPLIT_NAME} split)", y=1.05)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "predicted_vs_true_pd_scatter.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Figures saved to: {FIGURES_DIR.relative_to(ROOT)}")


## Part 9 - Stability (not yet implemented)

Left as a stub, per discussion. What still needs building, when we get to it:

- **Linear models**: `02`/`04`'s rolling-window refit loop needs to persist
  the fitted coefficients (and intercept) at each refit, not just the
  resulting predictions. Right now only `training_history` (loss curves) is
  saved, so a re-run against a modified refit loop is needed before
  coefficient drift can be tracked here.
- **MLPs**: since raw weights aren't comparable across independently fitted
  networks, stability needs each refit's MLP evaluated on a *fixed* reference
  set of macro-covariate inputs (not the rolling training window), then
  compared across refits (e.g. drift in predicted PD on that fixed set).
  This also requires a small addition to the refit loop to keep each
  intermediate model (or its predictions on the fixed set) rather than only
  the final ensemble.
- Both of the above are simulation-first, mirroring this notebook's split
  into calibration/discriminative-power now and stability later; the
  empirical notebook will follow the same structure once this is settled.
